# Smart MCQ Solver


## Overview

This notebook uses a Retrieval-Augmented Generation (RAG) pipeline to solve multiple-choice questions. The approach retrieves relevant information from a knowledge base first, and then an LLM picks the best answers based on that context.

The pipeline has these components:
- **Wikipedia corpus** - 190 scraped articles serving as the knowledge base
- **Training QA cards** - training questions formatted with their answers as reference documents, indexed alongside Wikipedia
- **FAISS vector index** - stores embeddings of all documents for fast similarity search
- **Cross-encoder reranker** - reranks the retrieved documents for better relevance
- **LLaMA-3-8B-Instruct** - the LLM that reads the retrieved context and ranks the answer options



## How the Pipeline Works

For each question, the pipeline does the following:

1. A search query is built using the question prompt + all 5 options
2. FAISS returns the 20 most similar chunks (could be Wikipedia text or training QA cards)
3. A cross-encoder reranks those 20 candidates, top 7 are kept
4. The top 7 contexts are fed to LLaMA-3, which ranks all 5 options (A-E)
5. The LLM output is parsed and the top 3 predictions are taken for submission

## 1. Install Libraries

LangChain is used for document handling and vector stores, FAISS for similarity search, sentence-transformers for the reranker, and transformers with bitsandbytes for running the LLM in 4-bit mode.

In [1]:
!pip install -qU wikipedia-api wikipedia
!pip install -qU langchain langchain-community langchain-huggingface langchain-core langchain-text-splitters
!pip install -qU faiss-gpu sentence-transformers
!pip install -qU transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 7.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.

In [2]:
import time
import os
import warnings
import logging
from transformers.utils import logging

warnings.filterwarnings('ignore')
logging.set_verbosity_error()

## 2. Data Scraping and Loading

The Wikipedia articles were scraped beforehand using the `wikipedia` Python library and stored as a separate **`wiki_pages`** dataset. Since the corpus has already been created, the scraping process is implemented in a separate script. This notebook only loads the pre-scraped Wikipedia dataset along with the competition CSV files.

In [3]:
wiki_folder = '/kaggle/input/datasets/sahilbind/wiki-pages'
train_csv_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
test_csv_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'

wiki_files = [f for f in os.listdir(wiki_folder) if f.endswith('.txt')]
print(f"Corpus: {len(wiki_files)} Wikipedia documents.")

Corpus: 190 Wikipedia documents.


## 3. Preparing the Knowledge Base

Two types of documents are prepared for the index:

1. **Wikipedia articles** - cleaned by removing markup like section headers, citation numbers, and URLs
2. **Training QA pairs** - each training question is formatted as a reference card with the prompt and the correct answer

Both types go into the same vector index so the retriever can pull up whichever is more relevant for a given test question.

In [4]:
import re
import pandas as pd
from langchain_core.documents import Document

def clean_wiki_text(text):
    """Remove Wikipedia markup like section headers, citations, URLs."""
    text = re.sub(r'^={3,}$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\[\d+\]\s*', '', text)
    text = re.sub(r'={2,}\s*(.*?)\s*={2,}', r'\1', text)
    text = re.sub(r'Source:\s*https?://\S+', '', text)
    text = re.sub(r'PageID:\s*\d+', '', text)
    text = re.sub(r'Wikipedia article:\s*', '', text)
    text = '\n'.join([line.strip() for line in text.split('\n')])
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.replace('\n', ' ').replace('\r', ' ').strip()

# Load Wikipedia corpus
corpus_docs = []
for filename in sorted(os.listdir(wiki_folder)):
    if filename.endswith('.txt'):
        filepath = os.path.join(wiki_folder, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            cleaned = clean_wiki_text(f.read())
            corpus_docs.append(Document(
                page_content=f"Article: {filename}\n\n{cleaned}",
                metadata={'source': filename, 'type': 'wikipedia'}
            ))

# Load training data and format as QA reference documents
train_df = pd.read_csv(train_csv_path)
qa_docs = []
for idx, row in train_df.iterrows():
    qa_text = (
        f"Question: {row['prompt']}\n"
        f"Answer: {row['answer']}"
    )
    qa_docs.append(Document(
        page_content=qa_text,
        metadata={'source': f"train_qa_{row['id']}", 'type': 'train_qa'}
    ))

print(f"Loaded {len(corpus_docs)} Wikipedia files and {len(qa_docs)} Training QA reference cards.")

Loaded 190 Wikipedia files and 2000 Training QA reference cards.


## 4. Chunking

Wikipedia articles are long, so they are split into smaller chunks of 700 characters with 200 characters of overlap between them. The overlap helps avoid losing information at chunk boundaries.

QA cards are kept as-is since they are already short and breaking them up would separate the question from its answer.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=200,
    separators=['. ', '? ', '! ', ', ', ' ', '']
)

wiki_chunks = text_splitter.split_documents(corpus_docs)
all_chunks = wiki_chunks + qa_docs

print(f"Total chunks indexed: {len(all_chunks)} (Wiki Chunks: {len(wiki_chunks)}, QA Cards: {len(qa_docs)})")

Total chunks indexed: 12535 (Wiki Chunks: 10535, QA Cards: 2000)


## 5. Embeddings and FAISS Vector Index

`BAAI/bge-base-en-v1.5` is used as the embedding model. It converts each text chunk into a 768-dimensional vector. All vectors are stored in a FAISS index for fast similarity search.

BGE models perform better when a specific prefix is added to the query during search.

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-base-en-v1.5',
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

print("Building FAISS vector index...")
vector_store = FAISS.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    distance_strategy=DistanceStrategy.COSINE
)

vector_store.save_local('/kaggle/working/unified_faiss_index')

print(f"FAISS index saved successfully containing {vector_store.index.ntotal} items.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building FAISS vector index...
FAISS index saved successfully containing 12535 items.


## 6. Cross-Encoder Reranker

FAISS returns the top 20 most similar chunks, but similarity search alone is not always accurate enough. A cross-encoder (`ms-marco-MiniLM-L-12-v2`) is used to rerank those 20 results.

Unlike the embedding model which encodes query and document separately, the cross-encoder processes both together and gives a more accurate relevance score. The top 7 results are kept after reranking.

In [7]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2', device='cuda')

def rerank_chunks(query, docs, top_k=7):
    pairs = [(query, doc.page_content) for doc in docs]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ranked[:top_k]]

print('Cross-encoder initialized.')

config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder initialized.


### 6.1 Retrieval Pipeline Validation

A helper cell for testing the retrieval pipeline with a sample query. This helps ensure that relevant document chunks are retrieved and correctly reranked before LLM-based answer generation.

In [8]:
# Input any sentence to test document retrieval
test_sentence = "What is the uncertainty principle in quantum mechanics?"

search_query = BGE_QUERY_PREFIX + test_sentence

# 1. Retrieve the top 20 matches from FAISS
retrieved_docs = vector_store.similarity_search(search_query, k=20)

# 2. Rerank using the cross-encoder and get the top 3
top_docs = rerank_chunks(test_sentence, retrieved_docs, top_k=3)

print(f"Query: {test_sentence}\n")
print(f"=== Top {len(top_docs)} Reranked Chunks ===")
for i, doc in enumerate(top_docs):
    doc_type = doc.metadata.get('type', 'unknown')
    source = doc.metadata.get('source', 'unknown')
    print(f"\n[{i+1}] Source: {source} (Type: {doc_type})")
    print("-" * 50)
    print(doc.page_content[:300] + "...")
    print("-" * 50)

Query: What is the uncertainty principle in quantum mechanics?

=== Top 3 Reranked Chunks ===

[1] Source: page_88.txt (Type: wikipedia)
--------------------------------------------------
Article: page_88.txt

Uncertainty principle  The uncertainty principle, also known as Heisenberg's indeterminacy principle, is a fundamental concept in quantum mechanics. It states that there is a limit to the precision with which certain pairs of physical properties, such as position and momentum, ...
--------------------------------------------------

[2] Source: page_88.txt (Type: wikipedia)
--------------------------------------------------
. From this point of view the uncertainty principle is not a fundamental quantum property but a concept "carried over from the language of our ancestors", as Kemble says.  Applications Since the uncertainty principle is such a basic result in quantum mechanics, typical experiments in quantum mechani...
--------------------------------------------------

[3] Sou

## 7. Load the Language Model

Meta's LLaMA-3-8B-Instruct is used as the generator. Since the full model is too large for Kaggle's GPU, it is loaded in 4-bit quantization using bitsandbytes. This reduces memory usage while keeping reasonable quality.

Greedy decoding (`do_sample=False`) is used so the outputs are the same every time.

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from kaggle_secrets import UserSecretsClient

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

model_id = 'meta-llama/Meta-Llama-3-8B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=hf_token,
    clean_up_tokenization_spaces=False
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    token=hf_token
)

llm_pipeline = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=20,
    do_sample=False,
    repetition_penalty=1.1,
    return_full_text=False
)

print('LLM configuration ready.')

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

LLM configuration ready.


## 8. RAG Solver Functions

This section defines the core functions of the pipeline:

- `parse_prediction` - extracts letter answers (A-E) from LLM output
- `build_retrieval_query` - combines the question and options into a search query
- `solve_mcq` - runs retrieval, reranking, and LLM inference for one question
- `calculate_map3` - computes MAP@3 for one question (1/rank if correct answer is in top 3, else 0)

In [10]:
import re
import numpy as np

def parse_prediction(raw_output):
    """Extract up to 3 unique answer labels from LLM output."""
    matches = re.findall(r'\b([A-E])\b', raw_output.strip())
    seen, found = set(), []
    for m in matches:
        if m not in seen:
            seen.add(m)
            found.append(m)
        if len(found) == 5:
            break
    # fill remaining with unused labels as fallback
    for label in ['A', 'B', 'C', 'D', 'E']:
        if label not in found:
            found.append(label)
        if len(found) == 5:
            break
    return found[:3]

def build_retrieval_query(row):
    opts = ' '.join(f"{l}. {row[l]}" for l in ['A', 'B', 'C', 'D', 'E'])
    return f"{row['prompt']} {opts}"

def solve_mcq(row, k_retrieve=20, k_rerank=7):
    """Run the full RAG pipeline for a single question."""
    raw_query = build_retrieval_query(row)
    search_query = BGE_QUERY_PREFIX + raw_query

    # Retrieve top candidates
    retrieved_docs = vector_store.similarity_search(search_query, k=k_retrieve)

    # Rerank retrieved docs
    top_docs = rerank_chunks(raw_query, retrieved_docs, top_k=k_rerank)

    context = '\n\n'.join(doc.page_content for doc in top_docs)
    options_text = '\n'.join(f"{l}. {row[l]}" for l in ['A', 'B', 'C', 'D', 'E'])

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert multiple-choice question solver. "
                "Use the provided context (which contains Wikipedia facts and reference QA pairs) "
                "to rank all 5 options from most likely correct to least likely correct. "
                "Output exactly 5 unique letters (A, B, C, D, E) separated by spaces in ranked order. "
                "Most confident answer first. No explanations. Example output: B A D C E"
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion:\n{row['prompt']}\n\nOptions:\n{options_text}\n\nRanked answer (all 5 letters):"
        }
    ]

    full_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = llm_pipeline(full_prompt)
    raw_answer = output[0]['generated_text']

    # check if a training QA card was in the retrieved context
    types_retrieved = [doc.metadata.get('type', 'unknown') for doc in top_docs]
    contains_qa = 'train_qa' in types_retrieved

    return parse_prediction(raw_answer), raw_answer, contains_qa

def calculate_map3(actual, predicted_list):
    for i, pred in enumerate(predicted_list[:3]):
        if pred == actual:
            return 1.0 / (i + 1)
    return 0.0

print('RAG Solver initialized.')

RAG Solver initialized.


## 9. Validation

Before running on the test set, the pipeline is validated on 100 random training samples to check performance. random_state=42 is used so the results are reproducible.

In [11]:
num_samples = 10
sample_df = train_df.sample(n=num_samples, random_state=42).reset_index(drop=True)

scores = []

for _, row in sample_df.iterrows():
    pred, _, _ = solve_mcq(row)   

    actual = row["answer"]
    score = calculate_map3(actual, pred)
    scores.append(score)

    print(f"Q{row['id']:3d} | Actual: {actual} | Pred: {' '.join(pred)} | MAP@3: {score:.2f}")

print(f"\n{'='*65}")
print(f"Overall Validation MAP@3 : {np.mean(scores):.4f}")
print(f"Top-1 Accuracy           : {sum(s == 1.0 for s in scores) / len(scores):.2%}")

Q1861 | Actual: D | Pred: D E C | MAP@3: 1.00
Q354 | Actual: E | Pred: E D A | MAP@3: 1.00
Q1334 | Actual: A | Pred: A E D | MAP@3: 1.00
Q906 | Actual: B | Pred: B E A | MAP@3: 1.00
Q1290 | Actual: C | Pred: C E D | MAP@3: 1.00
Q1274 | Actual: E | Pred: E D A | MAP@3: 1.00
Q939 | Actual: C | Pred: C E D | MAP@3: 1.00
Q1732 | Actual: A | Pred: A E D | MAP@3: 1.00
Q 66 | Actual: E | Pred: E D A | MAP@3: 1.00
Q1324 | Actual: A | Pred: A B C | MAP@3: 1.00

Overall Validation MAP@3 : 1.0000
Top-1 Accuracy           : 100.00%


## 10. Generate Submission

The pipeline is run on all test questions and predictions are saved as `submission.csv`.

In [12]:
test_df = pd.read_csv(test_csv_path)
print(f"Running test prediction for {len(test_df)} queries...")

test_predictions = []

for idx, row in test_df.iterrows():
    try:
        pred, _, _ = solve_mcq(row) 
    except Exception as e:
        print(f"Error on Q{row['id']}: {e}")
        pred = ['A', 'B', 'C']

    test_predictions.append(' '.join(pred))

    if (idx + 1) % 50 == 0:
        print(f"{idx + 1}/{len(test_df)} queries completed.")

print("\nPredictions completed.")

Running test prediction for 500 queries...
50/500 queries completed.
100/500 queries completed.
150/500 queries completed.
200/500 queries completed.
250/500 queries completed.
300/500 queries completed.
350/500 queries completed.
400/500 queries completed.
450/500 queries completed.
500/500 queries completed.

Predictions completed.


In [13]:
submission_df = pd.DataFrame({
    'ID': test_df['id'],
    'Prediction': test_predictions
})

submission_path = '/kaggle/working/submission.csv'
submission_df.to_csv(submission_path, index=False)

print(f"Submission saved to {submission_path}")
print(f"Shape: {submission_df.shape}")
print(f"\nFirst 5 rows:")
submission_df.head()

Submission saved to /kaggle/working/submission.csv
Shape: (500, 2)

First 5 rows:


,ID,Prediction
0,1,A E D
1,2,B E D
2,3,B E D
3,4,E D C
4,5,C E D
